# Wektorowa baza danych na przykładzie Qdrant

W celu zapisu embeddingów, należy użyć bazy danych, która wspiera przechowywanie wektorów. 

Przykładem takiej bazy danych jest Qdrant.

```
docker pull qdrant/qdrant

docker run -p 6333:6333 -p 6334:6334 \
    -v "$(pwd)/qdrant_storage:/qdrant/storage:z" \
    qdrant/qdrant
```

http://localhost:6333/dashboard

## Połączenie z bazą

In [ ]:
import numpy as np
from qdrant_client import models, QdrantClient
from qdrant_client.models import VectorParams, Distance

In [ ]:
client = QdrantClient(url="http://localhost:6333")
client

## Tworzenie kolekcji

In [ ]:
client.create_collection("test_collection", vectors_config=VectorParams(size=5, distance=Distance.COSINE))

## Dodawanie i modyfikacja rekordów

Aby dodać lub zmodyfikować rekordy w kolekcji używamy funkcji `upsert`.

Jeśli miarą odległości, której używamy będzie `cosine`, to dodane wektory ulegną normalizacji – długość wektora będzie jednostkowa, przy zachowaniu proporcji między składowymi.

**Dodawanie rekordów**

In [ ]:
points = [
    models.PointStruct(
        id=1, payload={"a": [1, 2, 3], "b": 30}, vector=[1, 2, 3, 4, 5]
    ),
    
    models.PointStruct(
        id=2, payload={"a": [10, 20, 30], "b": 40}, vector=np.array([10, 20, 30, 40, 50])
    ),
]

points

In [ ]:
client.upsert("test_collection", points=points)

**Modyfikacja rekordów**

Aby zmodyfikować rekordy, należy dodać rekordy o takim samym id.

W poniższym przykładzie rekord o id=2 zostanie zmodyfikowany. Przy okazji dodamy rekord o id=3.

In [ ]:
points = [
    models.PointStruct(
        id=2, payload={"a": [10, 20, 30], "b": 50}, vector=np.array([-10, -20, -30, -40, -50])
    ),
    
    models.PointStruct(
        id=3, payload={"a": {"list": [10, 20, 30], "string": "hello"}}, vector=[-101, -201, -301, -401, -501]
    ),
]

points

In [ ]:
client.upsert("test_collection", points=points)

## Wyszukiwanie podobnych wektorów

In [ ]:
search_results = client.query_points(
    collection_name="test_collection",
    query=[0.1, 0.2, 0.3, 0.4, 0.5],
    with_vectors=True,
    with_payload=False,
    limit=2
)

In [ ]:
search_results.points

In [ ]:
for vector in search_results.points:
    print(vector.vector)

## Usuwanie rekordów oraz kolekcji

### Usunięcie rekordu

In [ ]:
client.delete(
    collection_name="test_collection",
    points_selector=models.PointIdsList(points=[1]),
)

### Usunięcie kolekcji

In [ ]:
client.delete_collection("test_collection")

> **ZADANIA**

## Qdrant w LangChain

Aby zapisać do bazy Qdrant zembedowane chunki należy użyć klasy `QdrantVectorStore`.

Najpierw należy utworzyć kolekcję niezależnie od LangChaina a następnie utworzyć obiekt vectorstore.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

In [ ]:
embeddings = HuggingFaceEmbeddings(model="sdadas/st-polish-paraphrase-from-distilroberta")

In [ ]:
client.create_collection("knowledge_database",
                         vectors_config=VectorParams(size=768, distance=Distance.COSINE))

In [ ]:
vectorstore = QdrantVectorStore(
    client=client,
    collection_name="knowledge_database",
    embedding=embeddings,
)

In [ ]:
texts = [
    "Andrzej ma psa.",
    "Bartek ma kota.",
    "Czarek ma królika.",

    "Andrzej jeździ samochodem.",
    "Bartek jeździ rowerem.",
    "Czarek jeździ tramwajem.",

    "Andrzej lubi pływać.",
    "Bartek lubi biegać.",
    "Czarek lubi grać w szachy."
]

metadatas = [
    {"length": len(text), "person": text.split()[0]} for text in texts
]

In [ ]:
vectorstore.add_texts(texts, metadatas)

**Uwaga:** Zaletą tego podejścia jest to, że szybko można zembeddować chunki i dodać je do bazy. Ale jeśli z jakiegoś powodu zależy nam na rozdzieleniu tych dwóch etapów, to lepiej użyć innego sposobu.

> **ZADANIA**